# Vetores das fotos com o EmbeddingGemma 2 (Colab, GPU)

Sem GPU, o EmbeddingGemma 2 leva cerca de 20 s por foto: indexar as fotos do catálogo
levaria dias. Este notebook faz só essa parte numa GPU do Colab e devolve um arquivo
no formato do armazenamento local do mini-lcm:

    embeddinggemma-2__imagem.npz   (arrays "ids" e "vetores", float32, normalizados)

Depois de baixar, coloque o arquivo em `dados/embeddings/` e rode a avaliação com
`EMBEDDER=gemma2`. Os vetores de texto do Gemma (`embeddinggemma-2@titulo__texto.npz`)
já existem e são reaproveitados.

**Antes de começar:** Ambiente de execução → Alterar o tipo de ambiente de execução → GPU.

## 1. Conferir a GPU e instalar as bibliotecas

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# pillow e requests já vêm no Colab: atualizá-los só gera avisos de conflito com pacotes do próprio Colab
!pip install -q -U "sentence-transformers[image]" transformers torchvision

## 2. Enviar o catálogo

Envie o arquivo `dados/itens.jsonl` do projeto (cerca de 9 MB). Só o `id` e a `foto_url`
de cada produto são usados; as fotos são baixadas da Amazon aqui no Colab.

In [ ]:
import json
from google.colab import files

enviado = files.upload()  # escolha dados/itens.jsonl
nome = next(iter(enviado))
itens = [json.loads(l) for l in enviado[nome].decode("utf-8").splitlines() if l.strip()]
produtos = {i["id"]: i["foto_url"] for i in itens if i.get("foto_url")}
print(f"{len(itens)} produtos no arquivo, {len(produtos)} com foto")

## 3. Baixar as fotos

Em paralelo (esperar a rede não usa a GPU). As fotos ficam em `/content/fotos`: se a
sessão cair, rodar de novo só baixa as que faltam.

In [ ]:
import os, requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from io import BytesIO

os.makedirs("/content/fotos", exist_ok=True)
http = requests.Session()
http.headers["User-Agent"] = "mini-lcm/0.1"

def baixar(par):
    item_id, url = par
    caminho = f"/content/fotos/{item_id}.jpg"
    if os.path.exists(caminho):
        return item_id, caminho
    try:
        r = http.get(url, timeout=15)
        r.raise_for_status()
        Image.open(BytesIO(r.content)).convert("RGB").save(caminho, "JPEG", quality=92)
        return item_id, caminho
    except Exception as e:
        return item_id, None

with ThreadPoolExecutor(16) as pool:
    baixadas = dict(pool.map(baixar, produtos.items()))
fotos = {i: c for i, c in baixadas.items() if c}
print(f"{len(fotos)} fotos prontas, {len(baixadas) - len(fotos)} falharam")

## 4. Carregar o modelo

Sem o codificador de áudio (não usado). Em GPUs com bfloat16 (A100, L4) o modelo roda em
bfloat16; na T4 do Colab gratuito, em float32. float16 não é suportado pelo modelo.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

tipo = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
modelo = SentenceTransformer("google/embeddinggemma-2", device="cuda",
                             config_kwargs={"audio_config": None},
                             model_kwargs={"torch_dtype": tipo})
print("tipo numérico:", tipo)

## 5. Calcular os vetores

Cada foto entra sem prefixo de tarefa, como `{"text": "<|image|>", "image": [arquivo]}`
(cartão do modelo). A cada 500 fotos o progresso é salvo no seu Google Drive (`parcial_gemma_fotos.npz`).
Comece com um teste de 32 fotos para ver o tempo por foto. As fotos vão em lotes pequenos
(`LOTE = 4`) para caber na memória da T4.

In [ ]:
import gc, time, numpy as np

# Fotos por vez. Cada foto vira centenas de tokens, e a atenção compara todos com todos:
# com 32 fotos, a T4 (15 GB) fica sem memória. Se der "CUDA out of memory", diminua para 2.
LOTE = 4

ids = sorted(fotos)
def codificar(lista_ids, lote=LOTE):
    entradas = [{"text": "<|image|>", "image": [fotos[i]]} for i in lista_ids]
    m = modelo.encode(entradas, batch_size=lote, convert_to_numpy=True, show_progress_bar=False)
    m = np.asarray(m, dtype=np.float32)
    return m / np.clip(np.linalg.norm(m, axis=1, keepdims=True), 1e-12, None)

gc.collect(); torch.cuda.empty_cache()
t = time.time()
teste = codificar(ids[:32])
seg = (time.time() - t) / 32
print(f"{seg * 1000:.0f} ms por foto; estimativa para {len(ids)} fotos: {seg * len(ids) / 60:.0f} min")

In [ ]:
# O progresso vai para o seu Google Drive: se o Colab desconectar, /content é apagado,
# mas o arquivo no Drive fica, e rodar esta célula de novo retoma de onde parou.
from google.colab import drive
drive.mount("/content/drive")
PARCIAL = "/content/drive/MyDrive/parcial_gemma_fotos.npz"

vetores, feitos = [], []
if os.path.exists(PARCIAL):
    p = np.load(PARCIAL)
    feitos, vetores = list(p["ids"]), [p["vetores"]]
    print(f"retomando: {len(feitos)} já feitos")
ja_feitos = set(feitos)
restantes = [i for i in ids if i not in ja_feitos]
for inicio in range(0, len(restantes), 500):
    parte = restantes[inicio:inicio + 500]
    vetores.append(codificar(parte, lote=LOTE))
    feitos += parte
    np.savez(PARCIAL, ids=np.array(feitos), vetores=np.concatenate(vetores))
    print(f"{len(feitos)} / {len(ids)}")

## 6. Gravar e baixar

O nome do arquivo segue o padrão do armazenamento local: `<modelo>__<canal>.npz`.
Coloque-o em `dados/embeddings/` no projeto.

In [ ]:
p = np.load(PARCIAL)
np.savez("/content/embeddinggemma-2__imagem.npz", ids=p["ids"], vetores=p["vetores"].astype(np.float32))
print(p["vetores"].shape)
files.download("/content/embeddinggemma-2__imagem.npz")